# Inverse covariance estimation with MFCF-LoGo.

## Generate data

In [16]:
import numpy as np
from scipy import linalg

def generate_spd_precision(n=5, density=0.2, eps=1e-8, prng=None):
    A = prng.random(size=(n, n)) * 1000
    precision = (A + A.T) / 2.0  # make symmetric

    # Sparsify (keep zeros symmetric; don't zero the diagonal)
    mask = prng.uniform(size=(n, n)) < density
    mask = np.triu(mask, k=1)  # keep strictly upper triangle
    mask = mask + mask.T
    np.fill_diagonal(mask, False)
    precision[mask] = 0.0

    # Diagonal loading: preserve ALL off-diagonal zeros, ensure PD
    min_eig = np.linalg.eigvalsh(precision).min()
    if min_eig <= eps:
        precision += (-min_eig + eps) * np.eye(n)

    return precision

n_samples = 100
n_features = 1000
prng = np.random.RandomState(1)

prec = generate_spd_precision(n_features, density=0.8, prng=prng)
cov = linalg.inv(prec)
d = np.sqrt(np.diag(cov))
cov /= d
cov /= d[:, np.newaxis]
prec *= d
prec *= d[:, np.newaxis]

X = prng.multivariate_normal(np.zeros(n_features), cov, size=n_samples)

## Estimate the covariance and precision matrices

In [17]:
from sklearn.covariance import GraphicalLassoCV
from mfcf_logo import MFCFLoGoCV, MFCFLoGo
import time

emp_cov = np.dot(X.T, X) / n_samples

#model = GraphicalLassoCV()
#model.fit(X)
#gl_cov_ = model.covariance_
#gl_prec_ = model.precision_

# MFCFCV
start = time.time()
model = MFCFLoGo()
model.fit(X)
end = time.time()
print(end - start)
cov_ = model.covariance_
prec_ = model.precision_

0.15246987342834473


In [13]:
prec_

array([[3.83551105e+08, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.37578114e+08, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.19170743e+09, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        1.35030692e+06, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 1.85030245e+08, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 9.22155247e+08]],
      shape=(2500, 2500))